# Titanic with LightGBM (Colab — Kaggle 다운로드 버전)

Kaggle [Titanic](https://www.kaggle.com/c/titanic) 대회를 **LightGBM** 으로 푸는 기본 예제입니다. upura 님의 [Kaggle 튜토리얼 04 (LightGBM)](https://www.kaggle.com/code/sishihara/upura-kaggle-tutorial-04-lightgbm) 를 참고해 **Google Colab 독립 실행 + 최신 LightGBM(4.x)** 에 맞게 정리했습니다.

- 실행 시 **Kaggle API로 데이터를 직접 다운로드**합니다 (`train.csv`/`test.csv`).
- Kaggle API 토큰이 **노트북에 내장**되어 있어 별도 입력 없이 바로 실행됩니다.
- 모델: `LightGBM` (CPU로 충분, 가벼움).
- 마지막에 제출 파일 `submission.csv` 가 `/content` 에 생성됩니다.

> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.  
> (토큰 재발급/폐기: https://www.kaggle.com/settings/api)

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
# 필요한 라이브러리 설치
import sys, subprocess
for pkg in ["kaggle", "lightgbm", "numpy", "pandas", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os

# Kaggle API 토큰 (내장)
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"

print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 Titanic 데이터 다운로드

In [ ]:
import os, glob, zipfile

WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()

api.competition_download_files("titanic", path=WORK_DIR, quiet=False)
for zpath in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(WORK_DIR)
    os.remove(zpath)

print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])

## 2. 특징량 준비

학습/테스트 데이터를 합쳐 전처리합니다 (범주형 인코딩, 결측치 보정, 파생 변수).  
> 최신 pandas(Copy-on-Write) 에서 안전하도록 `df[col] = df[col].fillna(...)` 형태로 작성했습니다.

In [ ]:
import numpy as np
import pandas as pd

train = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
test = pd.read_csv(os.path.join(WORK_DIR, "test.csv"))
gender_submission = pd.read_csv(os.path.join(WORK_DIR, "gender_submission.csv"))

data = pd.concat([train, test], sort=False)

data["Sex"] = data["Sex"].map({"male": 0, "female": 1})
data["Embarked"] = data["Embarked"].fillna("S").map({"S": 0, "C": 1, "Q": 2}).astype(int)
data["Fare"] = data["Fare"].fillna(data["Fare"].mean())
data["Age"] = data["Age"].fillna(data["Age"].median())
data["FamilySize"] = data["Parch"] + data["SibSp"] + 1
data["IsAlone"] = (data["FamilySize"] == 1).astype(int)

delete_columns = ["Name", "PassengerId", "Ticket", "Cabin"]
data = data.drop(delete_columns, axis=1)

train = data[:len(train)].copy()
test = data[len(train):].copy()

y_train = train["Survived"]
X_train = train.drop("Survived", axis=1)
X_test = test.drop("Survived", axis=1)

X_train.head()

## 3. LightGBM 학습

검증용 데이터를 분리하고, 범주형 변수를 지정한 뒤 LightGBM 으로 학습합니다. `early_stopping` 콜백으로 과대적합을 방지합니다.

In [ ]:
from sklearn.model_selection import train_test_split
import lightgbm as lgb

X_tr, X_valid, y_tr, y_valid = train_test_split(
    X_train, y_train, test_size=0.3, random_state=0, stratify=y_train
)

# LightGBM 이 자동 처리할 범주형 변수
categorical_features = ["Embarked", "Pclass", "Sex"]

lgb_train = lgb.Dataset(X_tr, y_tr, categorical_feature=categorical_features)
lgb_eval = lgb.Dataset(X_valid, y_valid, reference=lgb_train, categorical_feature=categorical_features)

params = {"objective": "binary", "verbose": -1}

# lightgbm 4.x: verbose_eval/early_stopping_rounds 대신 콜백 사용
model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_train, lgb_eval],
    num_boost_round=1000,
    callbacks=[lgb.early_stopping(10), lgb.log_evaluation(10)],
)

## 4. 예측 & 제출 파일 생성

LightGBM 은 확률을 출력하므로, 0.5 를 기준으로 0/1 로 변환합니다.

In [ ]:
y_pred = model.predict(X_test, num_iteration=model.best_iteration)
y_pred = (y_pred > 0.5).astype(int)

submission_path = os.path.join(WORK_DIR, "submission.csv")
sub = gender_submission.copy()
sub["Survived"] = y_pred
sub.to_csv(submission_path, index=False)
print("Your submission was successfully saved to:", submission_path)
sub.head()

## 5. 제출 파일 내려받기

`submission.csv` 가 `/content` 에 생성되었습니다. 좌측 파일창에서 받거나 아래 셀로 바로 내려받으세요.

In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기

위에서 생성된 `submission.csv` 를 아래 페이지에서 업로드해 제출하세요:

👉 **[Titanic 제출 페이지](https://www.kaggle.com/c/titanic/submit)**

- 대회 홈: https://www.kaggle.com/c/titanic
> 제출하려면 해당 대회에 Join(규칙 동의)되어 있어야 합니다.